## Notebook 3: Analytical Models and Feature Engineering (F5)

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import Word2Vec

In [2]:
LIBRARY = pd.read_csv("../data/OutputTables/LIBRARY.csv", index_col="book_id")
TOKEN = pd.read_csv("../data/OutputTables/TOKEN.csv", index_col=["book_id", "chapter_num", "para_num", "sent_num"])
VOCAB = pd.read_csv("../data/OutputTables/VOCAB.csv")
BOW = pd.read_csv("../data/OutputTables/BOW.csv", index_col=0)
TFIDF = pd.read_csv("../data/OutputTables/TFIDF.csv", index_col=0)

TOKEN_WORDS = TOKEN[
    (TOKEN["is_alpha"] == True) &
    (TOKEN["is_stop"] == False)
].copy()

In [3]:
pca = PCA(n_components=5, random_state=42)
PC_matrix = pca.fit_transform(TFIDF)

DOC_PC = pd.DataFrame(
    PC_matrix,
    index=TFIDF.index,
    columns=[f"PC{i+1}" for i in range(5)]
)

DOC_PC.index.name = "book_id"
DOC_PC.head()

,PC1,PC2,PC3,PC4,PC5
book_id,,,,,
62,0.477926,0.095433,0.040472,0.141754,0.277880
64,0.531419,0.109949,0.053666,0.155574,0.257750
68,0.509700,0.111177,0.046263,0.151921,0.325791
72,0.358055,0.095634,0.013219,0.108719,0.414839
83,0.009755,-0.032405,-0.103933,-0.186323,-0.028651


In [4]:
PCA_LOADINGS = pd.DataFrame(
    pca.components_.T,
    index=TFIDF.columns,
    columns=[f"PC{i+1}" for i in range(5)]
)

PCA_LOADINGS.index.name = "term_str"
PCA_LOADINGS.head()

,PC1,PC2,PC3,PC4,PC5
term_str,,,,,
aaaah,-0.000086,-0.000054,-0.000063,0.000002,0.000016
aaagh,-0.000960,0.002489,0.000645,0.001075,-0.000646
aaah,-0.000102,-0.000093,-0.000061,0.000112,-0.000111
aaanthor,0.008790,0.003320,0.000561,0.004975,0.020270
aah,-0.000515,-0.000299,-0.000452,-0.000425,0.000461


In [5]:
PCA_VARIANCE = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(5)],
    "explained_variance_ratio": pca.explained_variance_ratio_
})

PCA_VARIANCE

,component,explained_variance_ratio
0,PC1,0.062358
1,PC2,0.044051
2,PC3,0.037535
3,PC4,0.033097
4,PC5,0.031398


In [6]:
DOC_PC.to_csv("../data/OutputTables/DOC_PC.csv")
PCA_LOADINGS.to_csv("../data/OutputTables/PCA_LOADINGS.csv")
PCA_VARIANCE.to_csv("../data/OutputTables/PCA_VARIANCE.csv", index=False)

In [7]:
lda = LatentDirichletAllocation(
    n_components=6,
    random_state=42,
    learning_method="batch"
)

DOC_TOPIC_matrix = lda.fit_transform(BOW)

DOC_TOPIC = pd.DataFrame(
    DOC_TOPIC_matrix,
    index=BOW.index,
    columns=[f"topic_{i}" for i in range(6)]
)

DOC_TOPIC.index.name = "book_id"
DOC_TOPIC.head()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5
book_id,,,,,,
62,0.000713,0.287723,0.711549,0.000005,0.000005,0.000005
64,0.000004,0.175508,0.824475,0.000004,0.000004,0.000004
68,0.000006,0.090270,0.909705,0.000006,0.000006,0.000006
72,0.000007,0.038823,0.961148,0.000007,0.000007,0.000007
83,0.000004,0.999981,0.000004,0.000004,0.000004,0.000004


In [8]:
TOPIC_TERM = pd.DataFrame(
    lda.components_,
    index=[f"topic_{i}" for i in range(6)],
    columns=BOW.columns
)

TOPIC_TERM.head()

,aaaah,aaagh,aaah,aaanthor,aah,ab,aback,abaddon,abandon,abandoned,...,zulma,zulthran,zurb,zydanowycz,à,ægri,æolian,æolus,œdipus,δ
topic_0,1.166561,1.166667,0.168803,0.166667,2.623548,0.166667,0.172191,0.167075,0.268189,17.929539,...,0.166667,0.166667,0.166667,0.167196,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
topic_1,0.166678,0.166735,0.166667,0.166724,0.166670,1.166667,1.256301,0.166667,29.089544,30.707401,...,1.166667,0.166667,0.166667,0.166667,1.166666,0.167345,1.166667,1.166667,0.167469,1.166667
topic_2,0.166667,0.166678,0.166667,32.166609,0.166667,0.166667,0.166667,0.166667,8.508447,10.904317,...,0.166667,0.166667,0.166703,0.166667,3.166667,0.166667,0.166667,0.166667,0.166667,0.166667
topic_3,0.166667,1.166587,0.166667,0.166667,0.708674,0.166667,0.166667,0.166667,1.151578,3.509997,...,0.166667,5.166667,26.166630,0.166669,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667
topic_4,0.166761,0.166667,0.166667,0.166667,0.166698,0.166667,0.168719,0.166667,1.769431,6.655191,...,0.166667,0.166667,0.166667,0.166667,0.166667,2.165988,0.166667,0.166667,1.165864,0.166667


In [9]:
terms = BOW.columns

for topic_name, row in TOPIC_TERM.iterrows():
    top_words = row.sort_values(ascending=False).head(12)
    print(topic_name)
    print(", ".join(top_words.index))
    print()

topic_0
one, would, said, like, get, could, know, thing, time, going, people, back

topic_1
would, could, one, u, said, upon, time, must, sea, day, two, water

topic_2
upon, one, could, would, man, toward, great, men, u, girl, warrior, eye

topic_3
one, said, vall, von, would, get, schlichten, time, well, like, could, know

topic_4
captain, nemo, ned, one, said, conseil, would, sir, like, could, know, time

topic_5
pencroft, would, harding, said, one, could, herbert, cyrus, engineer, time, ship, neb



In [10]:
DOC_TOPIC.to_csv("../data/OutputTables/DOC_TOPIC.csv")
TOPIC_TERM.to_csv("../data/OutputTables/TOPIC_TERM.csv")

In [11]:
sentences = (
    TOKEN_WORDS.reset_index()
    .sort_values(["book_id", "sent_num", "tok_num"])
    .groupby(["book_id", "sent_num"])["lemma"]
    .apply(list)
    .tolist()
)

len(sentences)

231

In [12]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    seed=42
)

pd_w2v_model = pd.DataFrame(
    [w2v_model.wv[word] for word in w2v_model.wv.index_to_key],
    index=w2v_model.wv.index_to_key
)

pd_w2v_model.index.name = "term_str"
pd_w2v_model.head()

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
would,0.121942,-0.050617,-0.490512,0.242048,-0.384079,0.370384,-0.012235,0.410503,-0.321796,0.068832,...,-0.693647,0.598838,0.002188,-0.462686,0.232806,-0.048393,0.106910,0.210693,-0.288130,-0.470996
one,0.191046,-0.270622,-0.387140,0.028854,-0.508792,0.109537,0.225567,0.372814,0.094545,-0.135521,...,-0.694782,0.563821,-0.218252,-0.491053,0.101900,-0.232188,0.007096,0.070013,-0.208703,-0.496969
could,0.142191,-0.049248,-0.515270,0.255620,-0.383497,0.319714,0.020537,0.433447,-0.308418,0.117119,...,-0.673396,0.574380,0.007364,-0.480027,0.211782,-0.021018,0.118645,0.206551,-0.288343,-0.444507
upon,0.235827,-0.391724,-0.463499,0.039444,-0.337405,-0.039321,0.071236,0.374062,0.055960,-0.051694,...,-0.721742,0.472464,-0.280851,-0.684553,-0.023770,-0.132083,-0.002127,-0.117708,-0.388085,-0.519357
said,0.190114,-0.092789,-0.579857,0.323386,-0.505782,0.312904,0.088458,0.403046,-0.167836,0.034654,...,-0.769683,0.707929,-0.062031,-0.474349,0.209058,-0.070265,0.097244,0.275532,-0.206290,-0.475865


In [13]:
pd_w2v_model.to_csv("../data/OutputTables/WORD2VEC.csv")

In [14]:
VOCAB = pd.read_csv("../data/OutputTables/VOCAB.csv", index_col="term_str")
VOCAB = VOCAB.join(PCA_LOADINGS, how="left")
VOCAB["dominant_topic"] = TOPIC_TERM.T.idxmax(axis=1)
VOCAB = VOCAB.join(pd_w2v_model, how="left")
VOCAB.to_csv("../data/OutputTables/VOCAB.csv")